In [56]:
# 필요한 패키지
# pip install sqlalchemy pymysql pandas

import pandas as pd
from sqlalchemy import create_engine, text

# ── DB 접속 정보 ───────────────────────────────────────────────────────────────
from DATA.stock_invest_function import get_db_host

db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

TABLE_NAME = "Korea_company_valuation_ver2"  # 스키마: investar.TABLE_NAME

def make_engine(db):
    url = (
        f"mysql+pymysql://{db['user']}:{db['password']}"
        f"@{db['host']}:{db['port']}/{db['database']}?charset=utf8mb4"
    )
    return create_engine(url, pool_pre_ping=True, future=True)

engine = make_engine(db_info)

# 1) forecast_date의 unique 값 추출
def get_unique_forecast_dates(include_null=False):
    q = f"""
        SELECT DISTINCT forecast_date
        FROM {TABLE_NAME}
        {"WHERE forecast_date IS NOT NULL" if not include_null else ""}
        ORDER BY forecast_date
    """
    with engine.begin() as conn:
        df = pd.read_sql(q, conn, parse_dates=["forecast_date"])
    return df["forecast_date"]

# 2) (ticker, forecast_date, keyword)로 indicator에 keyword가 포함된 값 조회 + date 기준 정렬
#    여러 indicator가 매칭되면 행으로 반환(롱 포맷). wide=True면 indicator별 칼럼으로 피벗.
def get_series_by_keyword(ticker, forecast_date, keyword, wide=False):
    sql = text(f"""
        SELECT `date`, `ticker`, `indicator`, `value`, `forecast_date`
        FROM {TABLE_NAME}
        WHERE ticker = :ticker
          AND forecast_date = :fdate
          AND indicator LIKE :kw
        ORDER BY `date`
    """)
    with engine.begin() as conn:
        df = pd.read_sql(
            sql, conn,
            params={"ticker": ticker, "fdate": forecast_date, "kw": f"%{keyword}%"},
            parse_dates=["date", "forecast_date"]
        )
    # 숫자형 보정
    if not df.empty:
        df["value"] = pd.to_numeric(df["value"], errors="coerce")
    if wide and not df.empty:
        df_wide = df.pivot_table(index="date", columns="indicator", values="value", aggfunc="last").sort_index()
        df_wide = df_wide.rename_axis(None, axis=1)
        return df_wide
    return df  # 롱 포맷: date, indicator, value …

# 3) indicator의 unique 값 추출
def get_unique_indicators(keyword=None):
    cond = "" if not keyword else "WHERE indicator LIKE :kw"
    sql = text(f"SELECT DISTINCT indicator FROM {TABLE_NAME} {cond} ORDER BY indicator")
    with engine.begin() as conn:
        df = pd.read_sql(sql, conn, params=(None if not keyword else {"kw": f"%{keyword}%"}))
    return df["indicator"]

# 4) (ticker, indicator, forecast_date 두 개) 입력 시 두 기간 차이 비교
#    반환: date 기준 병합(outer), col: value_fd1, value_fd2, diff = fd2 - fd1
def compare_indicator_between_dates(ticker, indicator, forecast_date_1, forecast_date_2):
    base_sql = text(f"""
        SELECT `date`, `value`
        FROM {TABLE_NAME}
        WHERE ticker = :ticker
          AND indicator = :indicator
          AND forecast_date = :fdate
        ORDER BY `date`
    """)
    with engine.begin() as conn:
        df1 = pd.read_sql(
            base_sql, conn,
            params={"ticker": ticker, "indicator": indicator, "fdate": forecast_date_1},
            parse_dates=["date"]
        )
        df2 = pd.read_sql(
            base_sql, conn,
            params={"ticker": ticker, "indicator": indicator, "fdate": forecast_date_2},
            parse_dates=["date"]
        )

    # 숫자형 보정
    for d in (df1, df2):
        if not d.empty:
            d["value"] = pd.to_numeric(d["value"], errors="coerce")

    df1 = df1.rename(columns={"value": f"value_{pd.to_datetime(forecast_date_1).date()}"})
    df2 = df2.rename(columns={"value": f"value_{pd.to_datetime(forecast_date_2).date()}"})

    out = pd.merge(df1, df2, on="date", how="outer").sort_values("date").set_index("date")
    if out.shape[1] == 2:
        cols = out.columns.tolist()
        out["diff"] = out[cols[1]] - out[cols[0]]  # fd2 - fd1
    return out

# ── 사용 예시 ─────────────────────────────────────────────────────────────────
# if __name__ == "__main__":
#     # 1) forecast_date 목록
#     print(get_unique_forecast_dates().tail())
#
#     # 2) 키워드로 조회 (롱/와이드)
#     ex_long = get_series_by_keyword(ticker="A005930", forecast_date="2025-10-26", keyword="revenue", wide=False)
#     ex_wide = get_series_by_keyword(ticker="A005930", forecast_date="2025-10-26", keyword="revenue", wide=True)
#     print(ex_long.head())
#     print(ex_wide.head())
#
#     # 3) indicator 유니크
#     print(get_unique_indicators().head())
#     # 특정 키워드만
#     print(get_unique_indicators(keyword="forecast").head())
#
#     # 4) 두 forecast_date 비교
#     comp = compare_indicator_between_dates(
#         ticker="A005930",
#         indicator="revenue_ensemble_forecast",   # 예: 정확한 indicator 이름 입력
#         forecast_date_1="2025-10-26",
#         forecast_date_2="2025-10-29"
#     )
#     print(comp.tail())



In [57]:
print(get_unique_forecast_dates().tail())

0   2025-10-26
1   2025-10-29
Name: forecast_date, dtype: datetime64[ns]


In [74]:
ex_long = get_series_by_keyword(ticker="A035420", forecast_date="2025-10-29", keyword="mc", wide=False)

In [75]:
ex_long

,date,ticker,indicator,value,forecast_date
0,2025-11-30,A035420,mc_sarima_noexog,4.647670e+10,2025-10-29
1,2025-11-30,A035420,mc_ets,4.422571e+10,2025-10-29
2,2025-11-30,A035420,mc_prophet,4.145374e+10,2025-10-29
3,2025-11-30,A035420,mc_lstm,6.111974e+10,2025-10-29
4,2025-11-30,A035420,mc_theta,5.248454e+10,2025-10-29
...,...,...,...,...,...
190,2026-11-30,A035420,mc_sarima_noexog,4.490099e+10,2025-10-29
191,2026-11-30,A035420,mc_ets,4.118665e+10,2025-10-29
192,2026-11-30,A035420,mc_prophet,3.096807e+10,2025-10-29
193,2026-11-30,A035420,mc_lstm,1.373340e+11,2025-10-29


In [76]:
# ex_long → indicator를 컬럼으로 피벗
ex_pivot = (
    ex_long
    .pivot_table(
        index=["date", "ticker"],      # 행 인덱스
        columns="indicator",           # 열로 변환할 컬럼
        values="value",                # 값으로 쓸 컬럼
        aggfunc="last"                 # 중복 시 마지막 값 사용
    )
    .reset_index()                     # date, ticker를 일반 컬럼으로 되돌림
)

# 필요 시 indicator 컬럼명 정리 (MultiIndex 제거)
ex_pivot.columns.name = None

# 확인
print(ex_pivot.head())


        date   ticker        mc_ets       mc_lstm    mc_prophet  \
0 2025-11-30  A035420  4.422571e+10  6.335000e+10  4.145374e+10   
1 2025-12-31  A035420  4.658527e+10  6.853977e+10  4.235223e+10   
2 2026-01-31  A035420  4.713317e+10  7.219876e+10  4.191304e+10   
3 2026-02-28  A035420  4.518903e+10  7.667797e+10  4.019087e+10   
4 2026-03-31  A035420  4.749873e+10  8.527390e+10  4.083852e+10   

   mc_sarima_noexog      mc_theta  
0      4.647670e+10  5.248454e+10  
1      4.821661e+10  5.169856e+10  
2      4.877225e+10  5.152333e+10  
3      4.680099e+10  5.134811e+10  
4      4.894327e+10  5.194530e+10  


In [77]:
ex_pivot.tail(14)

,date,ticker,mc_ets,mc_lstm,mc_prophet,mc_sarima_noexog,mc_theta
0,2025-11-30,A035420,4.422571e+10,6.335000e+10,4.145374e+10,4.647670e+10,5.248454e+10
1,2025-12-31,A035420,4.658527e+10,6.853977e+10,4.235223e+10,4.821661e+10,5.169856e+10
2,2026-01-31,A035420,4.713317e+10,7.219876e+10,4.191304e+10,4.877225e+10,5.152333e+10
3,2026-02-28,A035420,4.518903e+10,7.667797e+10,4.019087e+10,4.680099e+10,5.134811e+10
4,2026-03-31,A035420,4.749873e+10,8.527390e+10,4.083852e+10,4.894327e+10,5.194530e+10
5,2026-04-30,A035420,4.531611e+10,9.001660e+10,3.852107e+10,4.673992e+10,5.176743e+10
6,2026-05-31,A035420,4.740665e+10,9.498468e+10,3.853614e+10,4.885055e+10,5.158957e+10
7,2026-06-30,A035420,5.410272e+10,1.045310e+11,4.169816e+10,5.545250e+10,5.161954e+10
8,2026-07-31,A035420,5.706555e+10,1.106902e+11,4.207256e+10,6.074421e+10,5.144095e+10
9,2026-08-31,A035420,5.325698e+10,1.164023e+11,3.916089e+10,5.691642e+10,5.126236e+10
